In [103]:
import numpy as np 
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, LSTM, Dropout,SimpleRNN, GRU, Embedding,Input
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

In [109]:
sentences = [
    # 15 Positive Sentences (indices 0 to 14)
    "The movie was an absolute masterpiece with incredible acting.",
    "I had a fantastic experience and will definitely recommend this place.",
    "The service was exceptionally fast, friendly, and attentive.",
    "This new laptop exceeded all my expectations in performance.",
    "We enjoyed a delightful dinner with fresh, flavorful ingredients.",
    "She received a well-deserved promotion after her hard work.",
    "The customer support team resolved my issue quickly and politely.",
    "It was an inspiring conference full of brilliant new ideas.",
    "The garden looks stunning with all the colorful spring flowers.",
    "I am truly grateful for all the kindness and support shown today.",
    "The concert was full of energy and completely unforgettable.",
    "This hotel room was spotless, comfortable, and beautifully designed.",
    "The dessert was rich, creamy, and baked to perfection.",
    "Our team achieved record-breaking sales this quarter.",
    "It is always a genuine pleasure working alongside such talented peers.",

    # 15 Negative Sentences (indices 15 to 29)
    "The film was boring, predictable, and way too long.",
    "I had a terrible experience and will never return to this store.",
    "The waiter was rude and ignored our table for nearly an hour.",
    "The phone battery drains completely in less than two hours.",
    "The food arrived cold, greasy, and completely tasteless.",
    "The project was canceled due to poor planning and miscommunication.",
    "Customer service was unhelpful and kept transferring my call.",
    "The software crashed repeatedly and corrupted my saved files.",
    "The room was noisy, dirty, and smelled like stale smoke.",
    "I regret buying this product because it broke on the very first day.",
    "The flight was delayed for six hours with zero explanation given.",
    "The instructions were confusing and missing several key steps.",
    "The package arrived severely damaged with broken glass inside.",
    "The app has too many invasive ads and constantly freezes up.",
    "The event was disorganized, chaotic, and an utter waste of time."
]

# Corresponding binary labels for training (1 = Positive, 0 = Negative)
labels = [1] * 15 + [0] * 15
labels = np.array(labels)

In [110]:
# ==============================================================================
# 2. TOKENIZATION & DATA PREPARATION
# ==============================================================================
tokenizer = Tokenizer(num_words=2000, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)

# Vocabulary size for the Embedding layer (+1 for 0-index padding)
vocab_size = len(tokenizer.word_index) + 1

# Convert text to sequences
seq = tokenizer.texts_to_sequences(sentences)

# Compute max length safely using builtins.max
max_len = builtins.max(len(i) for i in seq)

# Build explicit NumPy arrays to prevent Keras cardinality/type errors
X_train = np.array(pad_sequences(seq, maxlen=max_len, padding='post'), dtype=np.int32)
y_train = np.array(labels, dtype=np.float32)

print(f"Vocabulary Size: {vocab_size}")
print(f"Max Sequence Length: {max_len}")
print(f"X_train Shape: {X_train.shape}")
print(f"y_train Shape: {y_train.shape}")

# ==============================================================================
# 3. FUNCTIONAL MODEL ARCHITECTURE
# ==============================================================================
emd_unit = 16
rnn_unit = 8

inp = Input(shape=(max_len,), dtype='int32', name='input')
emb = Embedding(input_dim=vocab_size, output_dim=emd_unit, mask_zero=True, name='embed')(inp)
rnn_out = SimpleRNN(rnn_unit, name='simplernn')(emb)
out = Dense(1, activation='sigmoid', name='output')(rnn_out)

model = Model(inputs=inp, outputs=out, name='Sentiment_RNN')
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

# ==============================================================================
# 4. MODEL TRAINING
# ==============================================================================
history = model.fit(
    X_train, 
    y_train, 
    epochs=25, 
    batch_size=8, 
    verbose=1
)

Vocabulary Size: 201
Max Sequence Length: 13
X_train Shape: (30, 13)
y_train Shape: (30,)


Model: "Sentiment_RNN"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 13)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed (Embedding)   │ (None, 13, 16)    │      3,216 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_11        │ (None, 13)        │          0 │ input[0][0]       │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simplernn           │ (None, 8)         │        200 │ embed[0][0],      │
│ (SimpleRNN)         │                   │            │ not_equal_11[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1)         │          9 │ simplernn[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,425 (13.38 KB)

 Trainable params: 3,425 (13.38 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.4333 - loss: 0.7181
Epoch 2/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6000 - loss: 0.6670 
Epoch 3/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7333 - loss: 0.6265 
Epoch 4/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9667 - loss: 0.5859 
Epoch 5/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 1.0000 - loss: 0.5461 
Epoch 6/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.5044 
Epoch 7/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.4640 
Epoch 8/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.4239 
Epoch 9/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.3837 
Epoch 10/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.3452 
Epoch 11/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.3092 
Epoch 12/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.2751 
Ep

In [111]:
# ==============================================================================
# 5. INFERENCE / TESTING ON NEW DATA
# ==============================================================================
test_sentences = [
    "The food was delicious and the staff was friendly.",
    "The app keeps crashing and it is completely useless."
]

test_seq = tokenizer.texts_to_sequences(test_sentences)
X_test = np.array(pad_sequences(test_seq, maxlen=max_len, padding='post'), dtype=np.int32)

predictions = model.predict(X_test)

print("\n--- Predictions ---")
for text, score in zip(test_sentences, predictions):
    sentiment = "Positive" if score[0] >= 0.5 else "Negative"
    print(f"Sentence: \"{text}\" -> Score: {score[0]:.4f} ({sentiment})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step

--- Predictions ---
Sentence: "The food was delicious and the staff was friendly." -> Score: 0.6533 (Positive)
Sentence: "The app keeps crashing and it is completely useless." -> Score: 0.4106 (Negative)
